In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np


def find_root(marker="data"):
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise FileNotFoundError(f"no parent of {Path.cwd()} contains '{marker}'")

ROOT = find_root()
sys.path.append(str(ROOT / "src"))

from preprocess import build_dataset
data=build_dataset()
print(len(data), "samples")
data.head()

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))

styles = {
    "Artesian Well": "black",
    "Thermal Spring": "grey",
    "Well": "blue"
}

for source_type, colour in styles.items():
    subset = data[data["Type"] == source_type]
    ax.scatter(subset["Longitude"], subset["Latitude"],
               c=colour, s=30, edgecolors="none",
               label=f"{source_type} (n={len(subset)})")

ax.set_title("Sample Locations")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(title="Type", frameon=False, fontsize=6.5, loc="upper right", 
          bbox_to_anchor = (1.3, 0.85))
ax.grid(alpha=0.5, linewidth=0.25)
plt.show()

In [ ]:
variations = ["Temperature", "Cond", "pH", "TDS",
              "Ca", "Mg", "Na", "K", "SiO2", "Depth"]

fig, axes = plt.subplots(2, 5, figsize=(15,5))

for ax, var in zip(axes.flat, variations):
    ax.boxplot(data[var].dropna(), vert=False, widths=0.5,
               flierprops=dict(marker="o", markersize=2.5,
                              markerfacecolor='black', alpha=0.5))
    ax.set_xlabel(var)
    ax.set_yticks([])
    ax.grid(axis="x", alpha=0.3, linewidth=0.5)

fig.suptitle("Box and Whiskers plots of Variables")
fig.tight_layout()
plt.show()


In [ ]:
def density(ax, series):
    values=series.to_numpy(dtype=float)
    if len(values) < 2 or np.allclose(values, values[0]):
        return
    bandwidth = 1.06 * values.std(ddof=1) * len(values) ** (-1 / 5)
    if bandwidth <= 0:
        return
    grid = np.linspace(values.min(), values.max(), 200)
    kernels = np.exp(-0.5*((grid[:, None] - values[None, :]) / bandwidth) ** 2)
    ax.plot(grid, kernels.sum(axis=1) / (len(values) * bandwidth * np.sqrt(2*np.pi)),
    c="black", linewidth=1)
    ax.set_xlim(-0.05, 1.05)

corr = data[variations].corr(method="pearson")

n = len(variations)
scaled=data[variations].copy()
span = (scaled.max() - scaled.min()).replace(0, np.nan)
scaled=((scaled-scaled.min()) / span).fillna(0.0)

fig, axes = plt.subplots(n, n, figsize=(14,14))

for i in range(n):
    for j in range(n):
        ax = axes[i, j]
        ax.set_xticks([]); ax.set_yticks([])

        if i == j:
            density(ax, scaled[variations[i]].dropna())
        elif i > j:
            ax.scatter(scaled[variations[j]], scaled[variations[i]],
                       s=8, c="red", alpha=0.75, edgecolors="black")
            ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
        else:
            r = corr.iloc[i, j]
            ax.text(0.5, 0.5, f"{r:.2f}", ha='center', va='center', fontsize=8+abs(r)*10, transform=ax.transAxes)
            ax.set_facecolor('#fafafa')

        if i == n - 1:
            ax.set_xlabel(variations[j], fontsize=9)
        if j == 0:
            ax.set_ylabel(variations[i], fontsize=9)

fig.suptitle("Correlation Matrix", y=1)
fig.tight_layout()
plt.show()

In [ ]:
skewed_variables = ["Cond", "TDS", "Ca", "Mg", "K", "SiO2"]

positives=data[skewed_variables].to_numpy(dtype=float)
offset=positives[positives>0].min() / 2

logged=np.log10(data[skewed_variables].clip(lower=0) + offset)

pd.DataFrame({
    "raw skew": data[skewed_variables].skew(),
    "log10 skew": logged.skew()
}).round(2)